# Глава 11. Метрики и матрица ошибок

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь построить и прочитать матрицу ошибок, рассчитать precision, recall и F1, объяснить их смысл и выбрать метрику, подходящую конкретной задаче.

Мы всё время мерили качество через accuracy — долю правильных ответов. Пора узнать важную и неприятную правду: одной accuracy категорически недостаточно, а иногда она откровенно врёт. И научиться понимать, какие именно ошибки делает модель, — потому что не все ошибки одинаково опасны.

## Провокация: 99% и полностью бесполезно

**Аналогия: детектор редкой болезни  💡**

Болезнь встречается у 1 человека из 100. Сделаем «модель», которая ВСЕМ говорит «здоров», не глядя ни на что. Её accuracy — 99%! Звучит прекрасно. Но она не нашла ни одного больного и абсолютно бесполезна — более того, опасна. Вот почему на несбалансированных данных (где одного класса намного больше) accuracy врёт: большой класс перекрывает всё, и высокая точность маскирует полную бесполезность.

## Матрица ошибок (confusion matrix)

Чтобы увидеть, где именно модель ошибается, строят матрицу ошибок — таблицу, которая раскладывает все ответы модели на четыре типа. Она отвечает не на вопрос «сколько правильно», а на вопрос «какие именно ошибки и как часто».

Модель сказала «да»

Модель сказала «нет»

На самом деле «да»

Верное попадание (TP)

Пропуск (FN)

На самом деле «нет»

Ложная тревога (FP)

Верный отказ (TN)

Два типа правильных ответов (по диагонали) и два типа ошибок. Пропуск (FN) — проглядели то, что было. Ложная тревога (FP) — забили тревогу зря. Ключевая идея: цена этих двух ошибок в разных задачах разная.

In [ ]:
# === Подготовка: выполни первой, если открыл новый блокнот ===
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "body_mass_g", "island", "species"])
X = df[["bill_length_mm", "body_mass_g", "island"]]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]),
     ["bill_length_mm", "body_mass_g"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["island"]),
])
full = Pipeline([("prep", prep), ("model", RandomForestClassifier(random_state=42))]).fit(X_train, y_train)

# Запускать в том же блокноте, где создан Pipeline "full" (глава 10),
# иначе будет ошибка NameError: имя 'full' не определено
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = full.predict(X_test)   # full — наш Pipeline из главы 10
cm = confusion_matrix(y_test, pred)
ConfusionMatrixDisplay(cm).plot()
plt.show()

## Precision, Recall и F1

Из матрицы выводятся три метрики, которые говорят гораздо больше, чем accuracy.

Метрика

Отвечает на вопрос

Когда важна

Precision

Из тех, кого модель назвала «да», сколько реально «да»?

Когда дорога ложная тревога

Recall

Из всех реальных «да» сколько модель нашла?

Когда дорого пропустить

F1

Баланс между precision и recall (их среднее)

Когда важно и то, и другое

**Аналогия: спам-фильтр против детектора болезни  💡**

В спам-фильтре важнее precision: лучше пропустить пару спам-писем во «Входящие», чем отправить важное письмо от учителя в спам (ложная тревога очень дорога). В детекторе болезни важнее recall: лучше лишний раз перепроверить здорового, чем пропустить больного (пропуск смертельно опасен). Одна и та же метрика accuracy не различает эти ситуации — а precision и recall различают.

In [ ]:
from sklearn.metrics import classification_report

# Сразу все метрики по каждому классу одной командой
print(classification_report(y_test, pred))

## Как выбрать главную метрику

Универсального ответа нет — всё зависит от задачи и цены ошибки. Прежде чем оценивать модель, задай себе главный вопрос: что дороже — ложная тревога или пропуск? Ответ и подскажет метрику. Если дороже пропустить (болезнь, мошенничество) — оптимизируй recall. Если дороже ложная тревога (спам, блокировка) — precision. Если важно и то и другое — F1. Умение осознанно выбрать метрику — это, пожалуй, главный признак зрелого специалиста по ML.

## Задания

Базовый уровень

Построй confusion matrix и classification_report для своего Pipeline на пингвинах. Сколько пропусков (FN) и сколько ложных тревог (FP)? Каких ошибок больше?

Со звёздочкой  ⭐

Для трёх задач — фильтр спама, детектор болезни, рекомендация фильма — реши, какая метрика (precision, recall или F1) главная и почему. Обоснуй через цену ошибки в каждой задаче.

Вопросы для обсуждения  💬

Почему accuracy 99% может означать полностью бесполезную модель?

Приведи задачу, где один пропуск (FN) страшнее сотни ложных тревог (FP).

Можно ли одновременно максимизировать и precision, и recall? Что обычно происходит, когда улучшаешь одно?

**Проверь себя  ℹ️**

1. Почему accuracy врёт на несбалансированных данных?

2. Чем precision отличается от recall?

3. Для детектора мошеннических операций какая метрика важнее и почему?